# RallyVision — Fine-tuning YOLOv8 para bola de beach tennis

**Antes de começar:**
1. Ativa a GPU: `Runtime > Change runtime type > T4 GPU`
2. Compacta a pasta `ml/spike/dataset/` em `dataset.zip` e faz upload para o teu Google Drive
3. Executa as células em ordem

In [ ]:
# Verifica se a GPU está disponível
import torch
print('GPU disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Modelo:', torch.cuda.get_device_name(0))
else:
    print('AVISO: sem GPU — vai em Runtime > Change runtime type > T4 GPU')

In [ ]:
!pip install ultralytics --quiet

In [ ]:
# Monta o Google Drive para acessar o dataset e salvar o modelo
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Ajusta o caminho se colocaste o zip numa pasta diferente do Drive
DATASET_ZIP = '/content/drive/MyDrive/dataset.zip'
DATASET_DIR = '/content/dataset'

if not os.path.exists(DATASET_ZIP):
    raise FileNotFoundError(
        f'Zip não encontrado em {DATASET_ZIP}\n'
        'Faz upload do dataset.zip para o Google Drive e ajusta o caminho acima.'
    )

!unzip -q "{DATASET_ZIP}" -d "{DATASET_DIR}"
!ls "{DATASET_DIR}"

In [ ]:
# Garante que o data.yaml aponta para os caminhos corretos no Colab
import yaml
from pathlib import Path

yaml_path = Path(DATASET_DIR) / 'data.yaml'
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg['train'] = str(Path(DATASET_DIR) / 'train' / 'images')
cfg['val']   = str(Path(DATASET_DIR) / 'train' / 'images')  # sem split separado
cfg.pop('test', None)

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print('data.yaml atualizado:')
print(open(yaml_path).read())

In [ ]:
import shutil
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data=str(yaml_path),
    epochs=100,
    batch=16,
    imgsz=640,
    name='ball',
    patience=20,
    optimizer='AdamW',
    lr0=1e-3,
    lrf=0.01,
    warmup_epochs=3,
    degrees=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.02,
    hsv_s=0.5,
    hsv_v=0.4,
    scale=0.5,
    perspective=0.0,
    save=True,
    plots=True,
)

In [ ]:
from pathlib import Path

best = Path(results.save_dir) / 'weights' / 'best.pt'
drive_out = '/content/drive/MyDrive/ball_yolo.pt'
shutil.copy(best, drive_out)

map50 = results.results_dict.get('metrics/mAP50(B)', 0)
map50_95 = results.results_dict.get('metrics/mAP50-95(B)', 0)

print('=' * 50)
print('TREINAMENTO CONCLUÍDO')
print(f'mAP50    : {map50:.3f}  (>0.70 é bom)')
print(f'mAP50-95 : {map50_95:.3f}')
print(f'Modelo salvo em: {drive_out}')
print('=' * 50)
print()
print('Próximo passo:')
print('  1. Baixa ball_yolo.pt do Google Drive')
print('  2. Coloca em ml/spike/')
print('  3. python yolo_ball_spike.py --video video.mp4 --weights ball_yolo.pt')